In [8]:
import pandas as pd
import numpy as np
import ast
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression


In [3]:
# Load dataset
df = pd.read_csv("reviews.csv")

# Convert ratings column (string → dictionary)
df["ratings"] = df["ratings"].apply(ast.literal_eval)

# Extract overall rating
df["overall"] = df["ratings"].apply(lambda x: x.get("overall"))

# Keep required columns only
df = df[["text", "overall"]]

# Remove missing values
df = df.dropna()

print("Dataset shape:", df.shape)


Dataset shape: (878561, 2)


In [4]:
def convert_sentiment(rating):
    if rating <= 2:
        return 0   # Negative
    elif rating == 3:
        return 1   # Neutral
    else:
        return 2   # Positive

df["sentiment"] = df["overall"].apply(convert_sentiment)

print(df["sentiment"].value_counts())

sentiment
2    642046
1    122565
0    113950
Name: count, dtype: int64


In [5]:
X = df["text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1,2),
        stop_words="english",
        min_df=5
    )),
    ("model", LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="lbfgs",
        multi_class="multinomial"
    ))
])


In [10]:
pipeline.fit(X_train, y_train)


c:\Users\Sami\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,steps,"[('tfidf', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [11]:
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.8019042415757514

Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.78      0.75     22790
           1       0.40      0.65      0.50     24513
           2       0.96      0.83      0.89    128410

    accuracy                           0.80    175713
   macro avg       0.70      0.76      0.71    175713
weighted avg       0.85      0.80      0.82    175713



In [12]:
joblib.dump(pipeline, "hotel_sentiment_model.pkl")

['hotel_sentiment_model.pkl']

In [13]:
model = joblib.load("hotel_sentiment_model.pkl")

In [15]:
review = "very bad experience, the room was dirty and the staff was rude"

prediction = model.predict([review])[0]

label_map = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

print("Sentiment:", label_map[prediction])

Sentiment: Negative
